In [1]:
import numpy as np
import pandas as pd
import polars as pl
import seaborn as sns

import matplotlib.pyplot as plt
from sklearn.tree import plot_tree
from sklearn.model_selection import train_test_split
from sksurv.ensemble import RandomSurvivalForest
from sksurv.linear_model import CoxPHSurvivalAnalysis
from sksurv.metrics import concordance_index_censored , concordance_index_ipcw
from sklearn.impute import SimpleImputer
from sksurv.util import Surv

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor




In [18]:
y_train = pl.read_csv("../data/raw/target_train.csv")   

y_train

ID,OS_YEARS,OS_STATUS
str,f64,f64
"""P132697""",1.115068,1.0
"""P132698""",4.928767,0.0
"""P116889""",2.043836,0.0
"""P132699""",2.476712,1.0
"""P132700""",3.145205,0.0
…,…,…
"""P121828""",null,null
"""P121829""",null,null
"""P121830""",1.99726,0.0


In [19]:
y_train.describe()

statistic,ID,OS_YEARS,OS_STATUS
str,str,f64,f64
"""count""","""3323""",3173.0,3173.0
"""null_count""","""0""",150.0,150.0
"""mean""",null,2.480713,0.504255
"""std""",null,2.588259,0.500061
"""min""","""P100000""",0.0,0.0
"""25%""",null,0.652055,0.0
"""50%""",null,1.652055,1.0
"""75%""",null,3.572603,1.0
"""max""","""P132729""",22.043836,1.0


In [20]:
def iqr_method(df , cl_lst):

    for (index , cl )  in enumerate(cl_lst):
        q1 = df.select(pl.col(cl).quantile(0.25)).item()
        q3 = df.select(pl.col(cl).quantile(0.75)).item()
        
        iqr = q3 - q1
        
        
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr

        
        # Winsorisation : capped values
        df = df.with_columns(
            pl.when(pl.col(cl) < lower_bound).then(lower_bound)
            .when(pl.col(cl) > upper_bound).then(upper_bound)
            .otherwise(pl.col(cl))
            .alias(cl)
        )
        
        
        
    return df

In [21]:
y_train = iqr_method(y_train , ["OS_YEARS"])

In [22]:
y_train.describe()

statistic,ID,OS_YEARS,OS_STATUS
str,str,f64,f64
"""count""","""3323""",3173.0,3173.0
"""null_count""","""0""",150.0,150.0
"""mean""",null,2.351739,0.504255
"""std""",null,2.133206,0.500061
"""min""","""P100000""",0.0,0.0
"""25%""",null,0.652055,0.0
"""50%""",null,1.652055,1.0
"""75%""",null,3.572603,1.0
"""max""","""P132729""",7.953425,1.0


In [23]:
def imputation_null_values(df , cl , estimator) :
    quant_vars = cl
    sub_df = df.select(quant_vars)

    # Convert to pandas
    sub_pd = sub_df.to_pandas()

    # Apply Model-Based Imputation (Random Forest)
    imputer = IterativeImputer(estimator=estimator, random_state=42)
    imputed_data = imputer.fit_transform(sub_pd)

    # 4. To Polars
    imputed_pl = pl.DataFrame(imputed_data, schema=sub_df.columns)

    # 5. Replace nulls
    df = df.with_columns([imputed_pl[col].alias(col) for col in quant_vars])
    
    return df


In [24]:
y_train = imputation_null_values(y_train, ["OS_YEARS"], estimator=RandomForestRegressor())


In [25]:
y_train.describe()

statistic,ID,OS_YEARS,OS_STATUS
str,str,f64,f64
"""count""","""3323""",3323.0,3173.0
"""null_count""","""0""",0.0,150.0
"""mean""",null,2.351739,0.504255
"""std""",null,2.084488,0.500061
"""min""","""P100000""",0.0,0.0
"""25%""",null,0.69863,0.0
"""50%""",null,1.808219,1.0
"""75%""",null,3.454795,1.0
"""max""","""P132729""",7.953425,1.0


In [26]:
y_train = y_train.with_columns(
    pl.col("OS_STATUS").fill_null(strategy="backward")
)

y_train

ID,OS_YEARS,OS_STATUS
str,f64,f64
"""P132697""",1.115068,1.0
"""P132698""",4.928767,0.0
"""P116889""",2.043836,0.0
"""P132699""",2.476712,1.0
"""P132700""",3.145205,0.0
…,…,…
"""P121828""",2.351739,0.0
"""P121829""",2.351739,0.0
"""P121830""",1.99726,0.0


In [27]:
y_train = y_train.with_columns(
    pl.col("OS_STATUS").cast(pl.Boolean).alias("OS_STATUS")
)

In [28]:
y_train["OS_STATUS"].value_counts()

OS_STATUS,count
bool,u32
false,1651
true,1672


In [29]:
y_train = y_train.rename(
    {"OS_YEARS" : "time" ,
     "OS_STATUS" : "event"}
)

In [30]:
from sksurv.util import Surv

y_train_pd = y_train.to_pandas()

y_train_pd = Surv.from_dataframe('event' , 'time' , y_train_pd)

In [31]:
y_train_pd

array([( True, 1.11506849), (False, 4.92876712), (False, 2.04383562), ...,
       (False, 1.99726027), ( True, 0.09589041), (False, 2.29041096)],
      shape=(3323,), dtype=[('event', '?'), ('time', '<f8')])